# TATR (Table Transformer) - Table Detection with LoRA Fine-tuning

## Objective
Train **Table Transformer (TATR)** for **Table Detection (TD)** task with architectural enhancements based on the Tablecert framework and LoRA fine-tuning.

This notebook implements:
1. **Baseline Inference** - Pre-trained TATR performance before any modifications
2. **Data Preparation** - Load COCO JSON annotations with augmentation
3. **Architecture Enhancement** - FreqFilter2D and Lite Transformer modules
4. **LoRA Fine-tuning** - Parameter-efficient training
5. **Evaluation & Comparison** - Before vs After enhancement

---

### Evaluation Criteria
- **Intersection over Union (IoU)** - Measures overlap between predicted and ground truth bounding boxes
- **Mean Average Precision (mAP)** - Detection accuracy across IoU thresholds (mAP@50, mAP@50-95)
- **F1-Score** - Harmonic mean of Precision and Recall

---

### Dataset Information
- **Images**: `/kaggle/input/datasets/mohammedahmedxx12/machathon-dataset/Phase1_Train_Dataset/images`
- **Annotations**: `/kaggle/input/datasets/philopaterg/stp26-preprocessed-dataset/preprocessed_dataset/table_detection`
- **Target Size**: 800×800 pixels
- **Classes**: 1 (table)

---

## Table of Contents
1. [Environment Setup](#setup)
2. [Global Configuration](#config)
3. [Baseline Inference (Before Enhancement)](#baseline)
4. [Data Preparation](#data)
5. [Architecture Modules (FreqFilter2D, Lite Transformer)](#modules)
6. [LoRA Fine-tuning Setup](#lora)
7. [Enhanced Model Training](#training)
8. [Evaluation Metrics Implementation](#evaluation)
9. [Comparison: Before vs After Enhancement](#comparison)

<a id='setup'></a>
## 1. Environment Setup

In [ ]:
# ============================================================
# ENVIRONMENT SETUP FOR TATR
# ============================================================
import subprocess
import sys

def install_packages():
    """Install required packages for TATR training with LoRA."""
    packages = [
        "transformers>=4.35.0",
        "torch>=2.0.0",
        "torchvision>=0.15.0",
        "Pillow>=9.0.0",
        "peft>=0.7.0",           # LoRA implementation
        "accelerate>=0.25.0",
        "albumentations>=1.3.0",
        "opencv-python-headless",
        "matplotlib",
        "seaborn",
        "pandas",
        "tqdm",
        "pycocotools",           # COCO evaluation tools
        "timm",                  # Image models (for backbone)
        "scipy",
    ]
    for pkg in packages:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
    print("✓ All packages installed successfully!")

install_packages()

In [ ]:
# ============================================================
# IMPORTS
# ============================================================
import os
import json
import shutil
import warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from tqdm import tqdm
from collections import defaultdict
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from typing import Dict, List, Tuple, Optional, Any
import cv2
import albumentations as A
from PIL import Image
import copy

# Transformers imports for TATR
from transformers import (
    DetrImageProcessor,
    TableTransformerForObjectDetection,
    TrainingArguments,
    Trainer,
    AutoConfig,
)
from transformers.image_transforms import center_to_corners_format

# PEFT for LoRA
from peft import (
    LoraConfig,
    get_peft_model,
    TaskType,
    PeftModel,
)

# Torch utilities
from torch.utils.data import Dataset, DataLoader

# Suppress warnings
warnings.filterwarnings('ignore')

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

<a id='config'></a>
## 2. Global Configuration

In [ ]:
# ============================================================
# GLOBAL CONFIGURATION
# ============================================================
CONFIG = {
    # Project Info
    "Project": "Tablecert_TD_TATR",
    "Version": "TATR_Enhanced_LoRA",
    "Base_Model": "microsoft/table-transformer-detection",
    
    # Paths (Kaggle)
    "Image_Dir": "/kaggle/input/datasets/mohammedahmedxx12/machathon-dataset/Phase1_Train_Dataset/images",
    "Annotation_Dir": "/kaggle/input/datasets/philopaterg/stp26-preprocessed-dataset/preprocessed_dataset/table_detection",
    "Augmentation_Config": "/kaggle/input/datasets/philopaterg/stp26-preprocessed-dataset/preprocessed_dataset/table_detection/augmentation_config.json",
    "Output_Dir": "/kaggle/working/tablecert_tatr_td",
    
    # Training Parameters
    # NOTE: Baseline is already excellent (F1=0.99), so use VERY conservative fine-tuning
    "Image_Size": 800,
    "Batch_Size": 4,           # Smaller batch for P100 (16GB VRAM) with transformer
    "Gradient_Accumulation": 4, # Effective batch size = 4 * 4 = 16
    "Epochs": 15,              # Reduced from 50 - baseline is near perfect
    "Learning_Rate": 1e-5,     # Reduced from 5e-5 - conservative fine-tuning
    "Weight_Decay": 1e-4,
    "Warmup_Ratio": 0.1,
    "Patience": 5,             # Reduced patience - stop early if no improvement
    "Device": "cuda" if torch.cuda.is_available() else "cpu",
    
    # TATR-specific Parameters
    "Num_Labels": 2,           # background + table (DETR style)
    "Num_Queries": 100,        # Number of object queries
    
    # LoRA Parameters (Based on Tablecert Paper Section 5.3)
    "LoRA_r": 8,               # Reduced from 16 - smaller changes
    "LoRA_alpha": 16,          # Reduced from 32
    "LoRA_dropout": 0.05,      # Reduced dropout
    "LoRA_target_modules": [   # Attention layers to apply LoRA
        "self_attn.q_proj",
        "self_attn.k_proj",
        "self_attn.v_proj",
        "self_attn.out_proj",
        "encoder_attn.q_proj",
        "encoder_attn.k_proj",
        "encoder_attn.v_proj",
        "encoder_attn.out_proj",
    ],
    
    # Architecture Enhancement Parameters (From Tablecert Paper)
    # NOTE: FreqFilter2D disabled - causes input distribution mismatch
    "Freq_Cutoff": 0.15,       # FreqFilter2D cutoff ratio (not used)
    "Arch_Lambdas": {
        "lambda_filter": 0.15,  # FreqFilter2D blending strength (not used)
        "lambda_lite": 0.1,     # Lite Transformer residual strength (not used)
    },
    
    # Evaluation Thresholds
    "IOU_Threshold": 0.5,
    "Conf_Threshold": 0.5,
}

# Create output directories
os.makedirs(CONFIG["Output_Dir"], exist_ok=True)
os.makedirs(f"{CONFIG['Output_Dir']}/baseline_results", exist_ok=True)
os.makedirs(f"{CONFIG['Output_Dir']}/enhanced_results", exist_ok=True)
os.makedirs(f"{CONFIG['Output_Dir']}/checkpoints", exist_ok=True)

print(f"Configuration loaded for {CONFIG['Version']}")
print(f"Device: {CONFIG['Device']}")
print(f"Image Size: {CONFIG['Image_Size']}x{CONFIG['Image_Size']}")
print(f"Batch Size: {CONFIG['Batch_Size']} (Effective: {CONFIG['Batch_Size'] * CONFIG['Gradient_Accumulation']})")
print(f"Learning Rate: {CONFIG['Learning_Rate']} (conservative for fine-tuning)")
print(f"LoRA Rank: {CONFIG['LoRA_r']}, Alpha: {CONFIG['LoRA_alpha']}")

---
## Evaluation Metrics Implementation

### 1. Intersection over Union (IoU)
$$\text{IoU} = \frac{\text{Area of Intersection}}{\text{Area of Union}} = \frac{|A \cap B|}{|A \cup B|}$$

### 2. Mean Average Precision (mAP)
- **mAP@50**: Average Precision at IoU threshold 0.50
- **mAP@50-95**: Average Precision across IoU thresholds from 0.50 to 0.95 (step 0.05)

### 3. F1-Score
$$F_1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$$

Where:
- $\text{Precision} = \frac{TP}{TP + FP}$
- $\text{Recall} = \frac{TP}{TP + FN}$

In [ ]:
# ============================================================
# EVALUATION METRICS CLASS
# ============================================================
class TableDetectionMetrics:
    """
    Comprehensive evaluation metrics for Table Detection.
    Implements IoU, mAP, Precision, Recall, and F1-Score.
    """
    
    def __init__(self, iou_threshold: float = 0.5):
        self.iou_threshold = iou_threshold
        self.reset()
    
    def reset(self):
        """Reset all metric counters."""
        self.all_predictions = []
        self.all_ground_truths = []
        self.tp = 0
        self.fp = 0
        self.fn = 0
        self.iou_scores = []
    
    @staticmethod
    def calculate_iou(box1: np.ndarray, box2: np.ndarray) -> float:
        """
        Calculate IoU between two bounding boxes.
        Boxes format: [x1, y1, x2, y2]
        """
        x1 = max(box1[0], box2[0])
        y1 = max(box1[1], box2[1])
        x2 = min(box1[2], box2[2])
        y2 = min(box1[3], box2[3])
        
        intersection = max(0, x2 - x1) * max(0, y2 - y1)
        
        area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
        area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
        union = area1 + area2 - intersection
        
        return intersection / union if union > 0 else 0.0
    
    def add_batch(self, predictions: List[np.ndarray], ground_truths: List[np.ndarray],
                  confidences: Optional[List[float]] = None):
        """
        Add a batch of predictions and ground truths for evaluation.
        
        Args:
            predictions: List of predicted boxes [x1, y1, x2, y2]
            ground_truths: List of ground truth boxes [x1, y1, x2, y2]
            confidences: Optional confidence scores for predictions
        """
        if confidences is None:
            confidences = [1.0] * len(predictions)
        
        # Store for mAP calculation
        self.all_predictions.extend(list(zip(predictions, confidences)))
        self.all_ground_truths.extend(ground_truths)
        
        # Match predictions to ground truths
        matched_gt = set()
        
        # Sort predictions by confidence
        pred_conf_pairs = sorted(zip(predictions, confidences), 
                                  key=lambda x: x[1], reverse=True)
        
        for pred_box, conf in pred_conf_pairs:
            best_iou = 0
            best_gt_idx = -1
            
            for gt_idx, gt_box in enumerate(ground_truths):
                if gt_idx in matched_gt:
                    continue
                iou = self.calculate_iou(pred_box, gt_box)
                if iou > best_iou:
                    best_iou = iou
                    best_gt_idx = gt_idx
            
            if best_iou >= self.iou_threshold and best_gt_idx != -1:
                self.tp += 1
                self.iou_scores.append(best_iou)
                matched_gt.add(best_gt_idx)
            else:
                self.fp += 1
        
        # Unmatched ground truths are false negatives
        self.fn += len(ground_truths) - len(matched_gt)
    
    def precision(self) -> float:
        """Calculate Precision = TP / (TP + FP)"""
        return self.tp / (self.tp + self.fp) if (self.tp + self.fp) > 0 else 0.0
    
    def recall(self) -> float:
        """Calculate Recall = TP / (TP + FN)"""
        return self.tp / (self.tp + self.fn) if (self.tp + self.fn) > 0 else 0.0
    
    def f1_score(self) -> float:
        """Calculate F1-Score = 2 * (Precision * Recall) / (Precision + Recall)"""
        p, r = self.precision(), self.recall()
        return 2 * p * r / (p + r) if (p + r) > 0 else 0.0
    
    def mean_iou(self) -> float:
        """Calculate mean IoU across all matched predictions."""
        return np.mean(self.iou_scores) if self.iou_scores else 0.0
    
    def calculate_ap(self, iou_thresh: float) -> float:
        """
        Calculate Average Precision at a specific IoU threshold.
        Uses 101-point interpolation (COCO style).
        """
        if not self.all_predictions or not self.all_ground_truths:
            return 0.0
        
        # Sort predictions by confidence
        sorted_preds = sorted(self.all_predictions, key=lambda x: x[1], reverse=True)
        
        tp_list = []
        fp_list = []
        gt_matched = set()
        
        for pred_box, conf in sorted_preds:
            best_iou = 0
            best_gt_idx = -1
            
            for gt_idx, gt_box in enumerate(self.all_ground_truths):
                if gt_idx in gt_matched:
                    continue
                iou = self.calculate_iou(pred_box, gt_box)
                if iou > best_iou:
                    best_iou = iou
                    best_gt_idx = gt_idx
            
            if best_iou >= iou_thresh and best_gt_idx != -1:
                tp_list.append(1)
                fp_list.append(0)
                gt_matched.add(best_gt_idx)
            else:
                tp_list.append(0)
                fp_list.append(1)
        
        # Cumulative sums
        tp_cumsum = np.cumsum(tp_list)
        fp_cumsum = np.cumsum(fp_list)
        
        # Precision and Recall curves
        precision_curve = tp_cumsum / (tp_cumsum + fp_cumsum)
        recall_curve = tp_cumsum / len(self.all_ground_truths)
        
        # 101-point interpolation
        recall_levels = np.linspace(0, 1, 101)
        precision_interp = np.zeros_like(recall_levels)
        
        for i, r in enumerate(recall_levels):
            precisions_at_r = precision_curve[recall_curve >= r]
            precision_interp[i] = np.max(precisions_at_r) if len(precisions_at_r) > 0 else 0
        
        return np.mean(precision_interp)
    
    def mAP_50(self) -> float:
        """Calculate mAP at IoU threshold 0.50"""
        return self.calculate_ap(0.50)
    
    def mAP_50_95(self) -> float:
        """Calculate mAP across IoU thresholds 0.50 to 0.95 (step 0.05)"""
        iou_thresholds = np.arange(0.50, 1.0, 0.05)
        aps = [self.calculate_ap(t) for t in iou_thresholds]
        return np.mean(aps)
    
    def get_all_metrics(self) -> Dict[str, float]:
        """Return all metrics as a dictionary."""
        return {
            "Precision": self.precision(),
            "Recall": self.recall(),
            "F1-Score": self.f1_score(),
            "Mean IoU": self.mean_iou(),
            "mAP@50": self.mAP_50(),
            "mAP@50-95": self.mAP_50_95(),
            "TP": self.tp,
            "FP": self.fp,
            "FN": self.fn
        }
    
    def print_summary(self, title: str = "Evaluation Results"):
        """Print formatted metric summary."""
        metrics = self.get_all_metrics()
        print(f"\n{'='*50}")
        print(f" {title}")
        print(f"{'='*50}")
        print(f" Precision:    {metrics['Precision']:.4f}")
        print(f" Recall:       {metrics['Recall']:.4f}")
        print(f" F1-Score:     {metrics['F1-Score']:.4f}")
        print(f" Mean IoU:     {metrics['Mean IoU']:.4f}")
        print(f" mAP@50:       {metrics['mAP@50']:.4f}")
        print(f" mAP@50-95:    {metrics['mAP@50-95']:.4f}")
        print(f"{'='*50}")
        print(f" TP: {metrics['TP']} | FP: {metrics['FP']} | FN: {metrics['FN']}")
        print(f"{'='*50}\n")
        return metrics

print("✓ TableDetectionMetrics class loaded")

<a id='baseline'></a>
## 3. Baseline Inference (Before Enhancement)

Run inference with the pre-trained TATR (Table Transformer) model **without any modifications** to establish baseline performance on our table detection dataset.

In [ ]:
# ============================================================
# LOAD TEST DATA FOR BASELINE EVALUATION
# ============================================================
def load_coco_annotations(json_path: str) -> Tuple[Dict, Dict]:
    """
    Load COCO format annotations.
    Returns image_info dict and annotations grouped by image_id.
    """
    with open(json_path, 'r') as f:
        data = json.load(f)
    
    # Build image lookup
    images = {img['id']: img for img in data['images']}
    
    # Group annotations by image_id (only table class, category_id=1)
    annotations = defaultdict(list)
    for ann in data['annotations']:
        if ann['category_id'] == 1:  # Table class only
            annotations[ann['image_id']].append(ann)
    
    return images, annotations

def coco_bbox_to_xyxy(bbox: List[float], img_width: int, img_height: int) -> np.ndarray:
    """
    Convert COCO bbox [x, y, w, h] to [x1, y1, x2, y2] (absolute pixels).
    """
    x, y, w, h = bbox
    return np.array([x, y, x + w, y + h])

# Load test set annotations
test_json_path = os.path.join(CONFIG["Annotation_Dir"], "test.json")
print(f"Loading test annotations from: {test_json_path}")

test_images, test_annotations = load_coco_annotations(test_json_path)
print(f"✓ Loaded {len(test_images)} test images with {sum(len(v) for v in test_annotations.values())} table annotations")

In [ ]:
# ============================================================
# BASELINE TATR MODEL INFERENCE
# ============================================================
def run_baseline_inference(model_name: str, test_images: Dict, test_annotations: Dict,
                           image_dir: str, conf_threshold: float = 0.5, 
                           device: str = "cuda") -> Tuple[Dict, Any, Any]:
    """
    Run inference with baseline TATR (Table Transformer) model.
    Returns evaluation metrics, model, and processor.
    """
    print(f"\n{'='*60}")
    print(" BASELINE TATR INFERENCE (Before Enhancement)")
    print(f"{'='*60}")
    
    # Load pre-trained TATR model and processor
    processor = DetrImageProcessor.from_pretrained(model_name)
    model = TableTransformerForObjectDetection.from_pretrained(model_name)
    model = model.to(device)
    model.eval()
    
    print(f"✓ Loaded model: {model_name}")
    print(f"✓ Model parameters: {sum(p.numel() for p in model.parameters()):,}")
    
    # Initialize metrics
    metrics = TableDetectionMetrics(iou_threshold=CONFIG["IOU_Threshold"])
    
    # Run inference on test set
    processed = 0
    
    with torch.no_grad():
        for img_id, img_info in tqdm(test_images.items(), desc="Running baseline inference"):
            img_path = os.path.join(image_dir, img_info['file_name'])
            
            if not os.path.exists(img_path):
                continue
            
            # Load and preprocess image
            image = Image.open(img_path).convert("RGB")
            inputs = processor(images=image, return_tensors="pt")
            inputs = {k: v.to(device) for k, v in inputs.items()}
            
            # Get ground truth boxes
            gt_anns = test_annotations.get(img_id, [])
            gt_boxes = [coco_bbox_to_xyxy(ann['bbox'], img_info['width'], img_info['height']) 
                        for ann in gt_anns]
            
            # Run inference
            outputs = model(**inputs)
            
            # Post-process predictions
            target_sizes = torch.tensor([[img_info['height'], img_info['width']]]).to(device)
            results = processor.post_process_object_detection(
                outputs, 
                target_sizes=target_sizes,
                threshold=conf_threshold
            )[0]
            
            pred_boxes = []
            pred_confs = []
            
            # Filter for table class (label 0 in TATR detection model)
            # Note: TATR detection model outputs label 0 for 'table'
            for score, label, box in zip(results["scores"], results["labels"], results["boxes"]):
                if label.item() == 0:  # Table class
                    pred_boxes.append(box.cpu().numpy())
                    pred_confs.append(float(score.cpu().numpy()))
            
            # Add to metrics
            metrics.add_batch(pred_boxes, gt_boxes, pred_confs)
            processed += 1
    
    print(f"\n✓ Processed {processed} images")
    
    # Get final metrics
    results = metrics.print_summary("BASELINE TATR Results")
    
    return results, model, processor

# Run baseline inference
print("Starting baseline evaluation...")
baseline_metrics, baseline_model, baseline_processor = run_baseline_inference(
    model_name=CONFIG["Base_Model"],
    test_images=test_images,
    test_annotations=test_annotations,
    image_dir=CONFIG["Image_Dir"],
    conf_threshold=CONFIG["Conf_Threshold"],
    device=CONFIG["Device"]
)

<a id='data'></a>
## 4. Data Preparation

Load COCO JSON annotations, create PyTorch Dataset class for TATR, and set up augmentation pipeline.

Since TATR natively uses COCO format, we don't need format conversion (unlike YOLO).

In [ ]:
# ============================================================
# LOAD AUGMENTATION CONFIGURATION
# ============================================================
def load_augmentation_config(config_path: str) -> Dict:
    """Load augmentation configuration from JSON file."""
    with open(config_path, 'r') as f:
        return json.load(f)

# Load augmentation config
aug_config = load_augmentation_config(CONFIG["Augmentation_Config"])
print("Augmentation Configuration:")
print(json.dumps(aug_config, indent=2)[:1500] + "...")

In [ ]:
# ============================================================
# BUILD ALBUMENTATIONS AUGMENTATION PIPELINE FOR TATR
# ============================================================
def build_augmentation_pipeline(config: Dict, train: bool = True) -> A.Compose:
    """
    Build Albumentations augmentation pipeline from config.
    Based on augmentation_config.json specifications.
    
    Args:
        config: Augmentation configuration dictionary
        train: If True, apply augmentations; if False, only resize
    """
    if not train:
        return A.Compose([
            A.Resize(CONFIG["Image_Size"], CONFIG["Image_Size"]),
        ], bbox_params=A.BboxParams(
            format='coco',
            min_area=100,
            min_visibility=0.3,
            label_fields=['category_ids']
        ))
    
    transforms = []
    
    # Geometric transforms
    geo = config.get('geometric', {})
    
    if geo.get('horizontal_flip', {}).get('enabled', False):
        transforms.append(A.HorizontalFlip(
            p=geo['horizontal_flip'].get('probability', 0.3)
        ))
    
    if geo.get('rotation', {}).get('enabled', False):
        transforms.append(A.Rotate(
            limit=geo['rotation'].get('limit_degrees', 5),
            border_mode=cv2.BORDER_CONSTANT,
            value=(255, 255, 255),
            p=geo['rotation'].get('probability', 0.2)
        ))
    
    if geo.get('perspective', {}).get('enabled', False):
        scale = geo['perspective'].get('scale', [0.02, 0.05])
        transforms.append(A.Perspective(
            scale=(scale[0], scale[1]),
            p=geo['perspective'].get('probability', 0.15)
        ))
    
    if geo.get('affine', {}).get('enabled', False):
        affine = geo['affine']
        transforms.append(A.Affine(
            scale=tuple(affine.get('scale', [0.95, 1.05])),
            translate_percent={'x': (-affine.get('translate_percent', 0.02), 
                                     affine.get('translate_percent', 0.02)),
                               'y': (-affine.get('translate_percent', 0.02), 
                                     affine.get('translate_percent', 0.02))},
            shear={'x': tuple(affine.get('shear', [-2, 2])),
                   'y': tuple(affine.get('shear', [-2, 2]))},
            p=affine.get('probability', 0.15)
        ))
    
    # Photometric transforms
    photo = config.get('photometric', {})
    
    if photo.get('brightness_contrast', {}).get('enabled', False):
        bc = photo['brightness_contrast']
        transforms.append(A.RandomBrightnessContrast(
            brightness_limit=bc.get('brightness_limit', 0.15),
            contrast_limit=bc.get('contrast_limit', 0.15),
            p=bc.get('probability', 0.4)
        ))
    
    if photo.get('hsv_shift', {}).get('enabled', False):
        hsv = photo['hsv_shift']
        transforms.append(A.HueSaturationValue(
            hue_shift_limit=hsv.get('hue_shift_limit', 10),
            sat_shift_limit=hsv.get('sat_shift_limit', 15),
            val_shift_limit=hsv.get('val_shift_limit', 15),
            p=hsv.get('probability', 0.3)
        ))
    
    if photo.get('grayscale', {}).get('enabled', False):
        transforms.append(A.ToGray(
            p=photo['grayscale'].get('probability', 0.1)
        ))
    
    if photo.get('blur', {}).get('enabled', False):
        transforms.append(A.GaussianBlur(
            blur_limit=photo['blur'].get('blur_limit', 3),
            p=photo['blur'].get('probability', 0.2)
        ))
    
    # Noise
    noise = photo.get('noise', {})
    if noise.get('gauss_noise', {}).get('enabled', False):
        gn = noise['gauss_noise']
        transforms.append(A.GaussNoise(
            var_limit=tuple(gn.get('var_limit', [5, 20])),
            p=gn.get('probability', 0.15)
        ))
    
    if photo.get('jpeg_compression', {}).get('enabled', False):
        jpeg = photo['jpeg_compression']
        transforms.append(A.ImageCompression(
            quality_lower=jpeg.get('quality_lower', 50),
            quality_upper=jpeg.get('quality_upper', 95),
            p=jpeg.get('probability', 0.2)
        ))
    
    # Document-specific
    doc = config.get('document_specific', {})
    if doc.get('grid_distortion', {}).get('enabled', False):
        transforms.append(A.GridDistortion(
            distort_limit=doc['grid_distortion'].get('distort_limit', 0.1),
            p=doc['grid_distortion'].get('probability', 0.1)
        ))
    
    # Add resize at the end to ensure consistent size
    transforms.append(A.Resize(CONFIG["Image_Size"], CONFIG["Image_Size"]))
    
    return A.Compose(
        transforms,
        bbox_params=A.BboxParams(
            format='coco',
            min_area=config.get('compose', {}).get('min_area', 100),
            min_visibility=config.get('compose', {}).get('min_visibility', 0.3),
            label_fields=['category_ids']
        )
    )

# Build augmentation pipelines
train_transform = build_augmentation_pipeline(aug_config, train=True)
val_transform = build_augmentation_pipeline(aug_config, train=False)
print(f"✓ Built training augmentation pipeline with {len(train_transform.transforms)} transforms")
print(f"✓ Built validation augmentation pipeline")

In [ ]:
# ============================================================
# TATR DATASET CLASS
# ============================================================
class TableDetectionDataset(Dataset):
    """
    PyTorch Dataset for Table Detection using TATR (Table Transformer).
    
    Loads images and annotations in COCO format, applies augmentations,
    and prepares inputs in the format expected by DETR/TATR models.
    """
    
    def __init__(self, 
                 json_path: str,
                 image_dir: str,
                 processor: DetrImageProcessor,
                 transform: Optional[A.Compose] = None):
        """
        Args:
            json_path: Path to COCO format JSON annotation file
            image_dir: Directory containing images
            processor: DETR/TATR image processor
            transform: Albumentations transform pipeline
        """
        self.image_dir = image_dir
        self.processor = processor
        self.transform = transform
        
        # Load COCO annotations
        with open(json_path, 'r') as f:
            coco_data = json.load(f)
        
        self.images = {img['id']: img for img in coco_data['images']}
        
        # Group annotations by image_id (only table class)
        self.annotations = defaultdict(list)
        for ann in coco_data['annotations']:
            if ann['category_id'] == 1:  # Table class only
                self.annotations[ann['image_id']].append(ann)
        
        # Create list of image ids that have at least one annotation
        self.image_ids = [
            img_id for img_id in self.images.keys() 
            if len(self.annotations.get(img_id, [])) > 0
        ]
        
        print(f"  Loaded {len(self.image_ids)} images with table annotations")
    
    def __len__(self) -> int:
        return len(self.image_ids)
    
    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        img_id = self.image_ids[idx]
        img_info = self.images[img_id]
        
        # Load image
        img_path = os.path.join(self.image_dir, img_info['file_name'])
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        # Get annotations for this image
        anns = self.annotations.get(img_id, [])
        
        # Prepare bboxes and labels in COCO format [x, y, width, height]
        bboxes = [ann['bbox'] for ann in anns]
        category_ids = [0] * len(bboxes)  # All tables are class 0 for TATR
        
        # Apply augmentation
        if self.transform is not None:
            try:
                transformed = self.transform(
                    image=image,
                    bboxes=bboxes,
                    category_ids=category_ids
                )
                image = transformed['image']
                bboxes = transformed['bboxes']
                category_ids = transformed['category_ids']
            except Exception as e:
                # If augmentation fails, just resize
                image = cv2.resize(image, (CONFIG["Image_Size"], CONFIG["Image_Size"]))
        
        # Convert to PIL for processor
        image_pil = Image.fromarray(image)
        
        # Get current image dimensions
        h, w = image.shape[:2] if len(image.shape) == 3 else (CONFIG["Image_Size"], CONFIG["Image_Size"])
        
        # Prepare target in DETR format
        # Convert COCO [x, y, w, h] to normalized [cx, cy, w, h]
        boxes_normalized = []
        for bbox in bboxes:
            x, y, bw, bh = bbox
            cx = (x + bw / 2) / w
            cy = (y + bh / 2) / h
            bw_norm = bw / w
            bh_norm = bh / h
            # Clamp to [0, 1]
            cx = max(0, min(1, cx))
            cy = max(0, min(1, cy))
            bw_norm = max(0, min(1, bw_norm))
            bh_norm = max(0, min(1, bh_norm))
            boxes_normalized.append([cx, cy, bw_norm, bh_norm])
        
        target = {
            "class_labels": torch.tensor(category_ids, dtype=torch.long),
            "boxes": torch.tensor(boxes_normalized, dtype=torch.float32) if boxes_normalized else torch.zeros((0, 4)),
        }
        
        # Process image
        encoding = self.processor(images=image_pil, return_tensors="pt")
        pixel_values = encoding["pixel_values"].squeeze(0)
        
        return {
            "pixel_values": pixel_values,
            "labels": target,
        }


def collate_fn(batch: List[Dict]) -> Dict[str, Any]:
    """Custom collate function for TATR training."""
    pixel_values = torch.stack([item["pixel_values"] for item in batch])
    labels = [item["labels"] for item in batch]
    return {
        "pixel_values": pixel_values,
        "labels": labels,
    }


# Create datasets
print("\nCreating datasets...")
processor = DetrImageProcessor.from_pretrained(CONFIG["Base_Model"])

train_dataset = TableDetectionDataset(
    json_path=os.path.join(CONFIG["Annotation_Dir"], "train.json"),
    image_dir=CONFIG["Image_Dir"],
    processor=processor,
    transform=train_transform
)

val_dataset = TableDetectionDataset(
    json_path=os.path.join(CONFIG["Annotation_Dir"], "val.json"),
    image_dir=CONFIG["Image_Dir"],
    processor=processor,
    transform=val_transform
)

test_dataset = TableDetectionDataset(
    json_path=os.path.join(CONFIG["Annotation_Dir"], "test.json"),
    image_dir=CONFIG["Image_Dir"],
    processor=processor,
    transform=val_transform
)

print(f"\n✓ Train dataset: {len(train_dataset)} images")
print(f"✓ Validation dataset: {len(val_dataset)} images")
print(f"✓ Test dataset: {len(test_dataset)} images")

<a id='modules'></a>
## 5. Architecture Modules (FreqFilter2D & Lite Transformer)

Implementation of architectural enhancements from the Tablecert paper adapted for TATR:
- **FreqFilter2D**: Frequency-domain filtering to suppress watermarks and noise (applied at backbone input)
- **Lite Transformer**: Lightweight contextual pre-processor for long-range dependency modeling

Based on Tablecert paper Section 5.3 (TATR architecture adaptation).

In [ ]:
# ============================================================
# FREQFILTER2D MODULE (Tablecert Paper - Appendix A)
# ============================================================
class FreqFilter2D(nn.Module):
    """
    Frequency Filter 2D Module (Tablecert Paper - Appendix A)
    
    Applies parameterized low-pass regularization in the frequency domain.
    Filters high-frequency noise (watermarks, scanning artifacts) before backbone.
    
    Formula A.1:
    X_filtered = Re(F^-1(F(X) * [(1-λ) + λ*M(r_cutoff)]))
    
    Where:
    - F: 2D FFT
    - M: Binary mask for low-frequency components
    - λ: Learnable filtering strength
    - r_cutoff: Cutoff ratio controlling bandwidth
    """
    
    def __init__(self, channels: int = 3, cutoff_ratio: float = 0.15, 
                 lambda_init: float = 0.15):
        super().__init__()
        self.cutoff_ratio = cutoff_ratio
        # Learnable parameter for blending strength
        self.lambda_filter = nn.Parameter(
            torch.tensor([lambda_init], dtype=torch.float32)
        )
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: Input tensor [B, C, H, W]
        Returns:
            Filtered tensor [B, C, H, W]
        """
        B, C, H, W = x.shape
        
        # 1. Apply 2D FFT
        x_fft = torch.fft.fft2(x.float(), norm="ortho")
        x_fft_shifted = torch.fft.fftshift(x_fft, dim=(-2, -1))
        
        # 2. Create low-pass mask
        cy, cx = H // 2, W // 2
        rh = int(H * self.cutoff_ratio / 2)
        rw = int(W * self.cutoff_ratio / 2)
        
        mask = torch.zeros((H, W), device=x.device, dtype=torch.float32)
        mask[max(0, cy - rh):min(H, cy + rh), 
             max(0, cx - rw):min(W, cx + rw)] = 1.0
        mask = mask.unsqueeze(0).unsqueeze(0)  # [1, 1, H, W]
        
        # 3. Apply parameterized filtering
        # Factor = (1 - λ) + λ * M
        lambda_clamped = torch.clamp(self.lambda_filter, 0.0, 1.0)
        factor = (1 - lambda_clamped) + (lambda_clamped * mask)
        
        # 4. Filter and reconstruct
        x_fft_filtered = x_fft_shifted * factor
        x_restored = torch.fft.ifft2(
            torch.fft.ifftshift(x_fft_filtered, dim=(-2, -1)), 
            norm="ortho"
        ).real
        
        return x_restored.to(x.dtype)


# ============================================================
# LITE TRANSFORMER MODULE (Tablecert Paper - Appendix E)
# ============================================================
class LiteTransformerBlock(nn.Module):
    """
    Lightweight Transformer Module (Tablecert Paper - Appendix E)
    
    Introduces long-range contextual modeling with minimal computational overhead.
    Operates as a residual refinement block.
    
    Formula A.10:
    Y = X + λ_lite * X_lite
    """
    
    def __init__(self, channels: int, num_heads: int = 4, 
                 mlp_ratio: float = 2.0, lambda_init: float = 0.1):
        super().__init__()
        self.channels = channels
        
        # Pointwise projection
        self.proj_in = nn.Conv2d(channels, channels, kernel_size=1)
        
        # Layer norm
        self.norm = nn.LayerNorm(channels)
        
        # Lightweight self-attention
        self.attn = nn.MultiheadAttention(
            embed_dim=channels,
            num_heads=num_heads,
            batch_first=True,
            dropout=0.1
        )
        
        # MLP
        hidden_dim = int(channels * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(channels, hidden_dim),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, channels),
            nn.Dropout(0.1),
        )
        
        # Output projection
        self.proj_out = nn.Conv2d(channels, channels, kernel_size=1)
        
        # Learnable blending parameter
        self.lambda_lite = nn.Parameter(
            torch.tensor([lambda_init], dtype=torch.float32)
        )
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: Input tensor [B, C, H, W]
        Returns:
            Refined tensor [B, C, H, W]
        """
        B, C, H, W = x.shape
        identity = x
        
        # Project input
        x_proj = self.proj_in(x)  # [B, C, H, W]
        
        # Reshape for attention: [B, H*W, C]
        x_seq = x_proj.flatten(2).transpose(1, 2)  # [B, H*W, C]
        
        # Apply layer norm
        x_norm = self.norm(x_seq)
        
        # Self-attention
        x_attn, _ = self.attn(x_norm, x_norm, x_norm)
        x_seq = x_seq + x_attn
        
        # MLP
        x_seq = x_seq + self.mlp(self.norm(x_seq))
        
        # Reshape back: [B, C, H, W]
        x_out = x_seq.transpose(1, 2).reshape(B, C, H, W)
        
        # Output projection
        x_lite = self.proj_out(x_out)
        
        # Residual blend with learnable λ
        lambda_clamped = torch.clamp(self.lambda_lite, 0.0, 1.0)
        return identity + lambda_clamped * x_lite


print("✓ FreqFilter2D module loaded")
print("✓ LiteTransformerBlock module loaded")

In [ ]:
# ============================================================
# ENHANCED TATR MODEL WITH ARCHITECTURAL MODIFICATIONS
# ============================================================
class EnhancedTableTransformer(nn.Module):
    """
    Enhanced Table Transformer (TATR) with Tablecert architectural modifications.
    
    Based on Tablecert paper Section 5.3:
    1. FreqFilter2D: Applied at backbone input (before ResNet)
    2. LiteTransformer: Applied to encoder features
    
    The model wraps the original TableTransformerForObjectDetection and adds
    enhancement modules while preserving the original forward pass structure.
    """
    
    def __init__(self, 
                 base_model: TableTransformerForObjectDetection,
                 freq_cutoff: float = 0.15,
                 lambda_filter: float = 0.15,
                 lambda_lite: float = 0.1,
                 apply_freq_filter: bool = True,
                 apply_lite_transformer: bool = True):
        super().__init__()
        
        self.base_model = base_model
        self.config = base_model.config
        self.apply_freq_filter = apply_freq_filter
        self.apply_lite_transformer = apply_lite_transformer
        
        # FreqFilter2D - applied to input images (3 channels)
        if apply_freq_filter:
            self.freq_filter = FreqFilter2D(
                channels=3,
                cutoff_ratio=freq_cutoff,
                lambda_init=lambda_filter
            )
            print(f"  ✓ FreqFilter2D injected (cutoff={freq_cutoff}, λ={lambda_filter})")
        
        # LiteTransformer - applied to backbone features
        # ResNet50 output channels at different stages
        if apply_lite_transformer:
            # Get the hidden dimension from the model config
            hidden_dim = base_model.config.d_model  # Usually 256
            self.lite_transformer = LiteTransformerBlock(
                channels=hidden_dim,
                num_heads=4,
                mlp_ratio=2.0,
                lambda_init=lambda_lite
            )
            print(f"  ✓ LiteTransformer injected (λ={lambda_lite})")
    
    def forward(self, pixel_values, labels=None, **kwargs):
        """
        Forward pass with architectural enhancements.
        
        Args:
            pixel_values: Input images [B, 3, H, W]
            labels: Optional target labels for training
            **kwargs: Additional arguments
        
        Returns:
            Model outputs (loss if labels provided, else predictions)
        """
        # 1. Apply FreqFilter2D to input images
        if self.apply_freq_filter:
            pixel_values = self.freq_filter(pixel_values)
        
        # 2. Get backbone features
        # Access the backbone through the model
        backbone_output = self.base_model.model.backbone(pixel_values)
        
        # Get the last feature map
        feature_maps = list(backbone_output.values())
        src = feature_maps[-1]  # [B, C, H, W]
        
        # 3. Apply LiteTransformer to backbone features
        if self.apply_lite_transformer:
            src = self.lite_transformer(src)
        
        # 4. Continue with the rest of the model
        # Input projection
        src = self.base_model.model.input_projection(src)
        
        # Get mask (all False for valid regions)
        B, C, H, W = src.shape
        mask = torch.zeros((B, H, W), device=src.device, dtype=torch.bool)
        
        # Position encoding
        pos_embed = self.base_model.model.position_encoding(src, mask)
        
        # Flatten spatial dimensions
        src = src.flatten(2).transpose(1, 2)  # [B, H*W, C]
        pos_embed = pos_embed.flatten(2).transpose(1, 2)  # [B, H*W, C]
        mask = mask.flatten(1)  # [B, H*W]
        
        # Query embeddings
        query_embeds = self.base_model.model.query_position_embeddings.weight.unsqueeze(0).repeat(B, 1, 1)
        
        # Transformer
        encoder_output = self.base_model.model.encoder(
            inputs_embeds=src,
            attention_mask=~mask,  # Transformers expect True for valid, DETR uses False
            position_embeddings=pos_embed,
        )
        
        decoder_output = self.base_model.model.decoder(
            inputs_embeds=torch.zeros_like(query_embeds),
            attention_mask=None,
            encoder_hidden_states=encoder_output.last_hidden_state,
            encoder_attention_mask=~mask,
            position_embeddings=query_embeds,
            query_position_embeddings=query_embeds,
        )
        
        # Get final hidden state
        hidden_states = decoder_output.last_hidden_state
        
        # Classification and box regression heads
        logits = self.base_model.class_labels_classifier(hidden_states)
        pred_boxes = self.base_model.bbox_predictor(hidden_states).sigmoid()
        
        loss = None
        if labels is not None:
            # Compute loss using the model's loss function
            outputs = {"logits": logits, "pred_boxes": pred_boxes}
            loss_dict = self.base_model.loss_function(
                outputs, labels, self.base_model.config.auxiliary_loss
            )
            loss = sum(loss_dict.values())
        
        return {
            "loss": loss,
            "logits": logits,
            "pred_boxes": pred_boxes,
        }


def create_enhanced_tatr_model(config: Dict, device: str = "cuda") -> EnhancedTableTransformer:
    """
    Create enhanced TATR model with architectural modifications.
    
    Args:
        config: Configuration dictionary
        device: Device to load model on
    
    Returns:
        EnhancedTableTransformer model
    """
    print("\n" + "="*60)
    print(" CREATING ENHANCED TATR MODEL")
    print("="*60)
    
    # Load base model
    base_model = TableTransformerForObjectDetection.from_pretrained(
        config["Base_Model"]
    )
    print(f"✓ Loaded base model: {config['Base_Model']}")
    
    # Create enhanced model
    enhanced_model = EnhancedTableTransformer(
        base_model=base_model,
        freq_cutoff=config["Freq_Cutoff"],
        lambda_filter=config["Arch_Lambdas"]["lambda_filter"],
        lambda_lite=config["Arch_Lambdas"]["lambda_lite"],
        apply_freq_filter=True,
        apply_lite_transformer=True,
    )
    
    enhanced_model = enhanced_model.to(device)
    
    # Count parameters
    total_params = sum(p.numel() for p in enhanced_model.parameters())
    trainable_params = sum(p.numel() for p in enhanced_model.parameters() if p.requires_grad)
    
    print(f"\n  Total Parameters: {total_params:,}")
    print(f"  Trainable Parameters: {trainable_params:,}")
    print("="*60 + "\n")
    
    return enhanced_model

# Test architecture enhancement (will be replaced with LoRA version)
print("Architecture enhancement modules ready")
print("Note: Full model will be created with LoRA in the next section")

<a id='lora'></a>
## 6. LoRA Fine-tuning Setup

Set up Low-Rank Adaptation (LoRA) for parameter-efficient fine-tuning of TATR.

Based on Tablecert paper Section 2.3 and 5.3:
- LoRA applies low-rank decomposition to attention layers
- Reduces trainable parameters significantly
- Maintains model performance while improving training efficiency

In [ ]:
# ============================================================
# SIMPLIFIED TRAINING WITH LORA (NO FREQFILTER2D)
# ============================================================
# NOTE: FreqFilter2D was causing input distribution mismatch.
# The pre-trained TATR was trained on unfiltered images.
# With only ~2% trainable params (LoRA), the model cannot adapt
# to FFT-transformed inputs. Removing FreqFilter2D for now.

def setup_lora_training(config: Dict) -> Tuple[nn.Module, DetrImageProcessor]:
    """
    Set up TATR model with LoRA for parameter-efficient fine-tuning.
    
    Based on Tablecert paper Section 5.3:
    - Apply LoRA to attention layers in encoder and decoder
    - Freeze base model weights
    
    Returns:
        Tuple of (model with LoRA, processor)
    """
    print("\n" + "="*60)
    print(" SETTING UP TATR WITH LoRA FINE-TUNING")
    print("="*60)
    
    # Load base model
    base_model = TableTransformerForObjectDetection.from_pretrained(
        config["Base_Model"]
    )
    processor = DetrImageProcessor.from_pretrained(config["Base_Model"])
    
    print(f"✓ Loaded base model: {config['Base_Model']}")
    
    # Configure LoRA
    # Target modules in TATR/DETR architecture
    target_modules = [
        "self_attn.k_proj",
        "self_attn.v_proj",
        "self_attn.q_proj",
        "self_attn.out_proj",
        "encoder_attn.k_proj",
        "encoder_attn.v_proj",
        "encoder_attn.q_proj",
        "encoder_attn.out_proj",
    ]
    
    lora_config = LoraConfig(
        r=config["LoRA_r"],
        lora_alpha=config["LoRA_alpha"],
        target_modules=target_modules,
        lora_dropout=config["LoRA_dropout"],
        bias="none",
        task_type=TaskType.FEATURE_EXTRACTION,  # Object detection
    )
    
    print(f"\nLoRA Configuration:")
    print(f"  - Rank (r): {config['LoRA_r']}")
    print(f"  - Alpha: {config['LoRA_alpha']}")
    print(f"  - Dropout: {config['LoRA_dropout']}")
    print(f"  - Target modules: {len(target_modules)} layer types")
    
    # Apply LoRA to base model
    lora_model = get_peft_model(base_model, lora_config)
    
    # Move to device
    lora_model = lora_model.to(config["Device"])
    
    # Count parameters
    total_params = sum(p.numel() for p in lora_model.parameters())
    trainable_params = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)
    
    print(f"\n  Total Parameters: {total_params:,}")
    print(f"  Trainable Parameters: {trainable_params:,}")
    print(f"  Trainable %: {100 * trainable_params / total_params:.2f}%")
    
    # Print trainable modules
    print("\n  Trainable modules:")
    lora_model.print_trainable_parameters()
    
    print("="*60 + "\n")
    
    return lora_model, processor


# Set up model with LoRA (no FreqFilter2D wrapper)
enhanced_model, processor = setup_lora_training(CONFIG)

In [ ]:
# ============================================================
# TRAINING LOOP FOR TATR WITH LORA
# ============================================================

class TATRTrainer:
    """
    Custom trainer for TATR with LoRA fine-tuning.
    
    Uses the model's built-in DETR loss with Hungarian matching
    instead of custom loss to ensure proper training.
    """
    
    def __init__(self, 
                 model: nn.Module,
                 train_dataset: Dataset,
                 val_dataset: Dataset,
                 config: Dict):
        self.model = model
        self.config = config
        self.device = config["Device"]
        
        # DataLoaders
        self.train_loader = DataLoader(
            train_dataset,
            batch_size=config["Batch_Size"],
            shuffle=True,
            collate_fn=collate_fn,
            num_workers=2,
            pin_memory=True,
        )
        
        self.val_loader = DataLoader(
            val_dataset,
            batch_size=config["Batch_Size"],
            shuffle=False,
            collate_fn=collate_fn,
            num_workers=2,
            pin_memory=True,
        )
        
        # Optimizer with LOWER learning rate for fine-tuning
        # Since baseline is already excellent, use conservative LR
        effective_lr = config["Learning_Rate"]
        
        self.optimizer = torch.optim.AdamW(
            [p for p in model.parameters() if p.requires_grad],
            lr=effective_lr,
            weight_decay=config["Weight_Decay"],
        )
        
        # Learning rate scheduler with warmup
        num_training_steps = len(self.train_loader) * config["Epochs"]
        
        self.scheduler = torch.optim.lr_scheduler.OneCycleLR(
            self.optimizer,
            max_lr=effective_lr,
            total_steps=num_training_steps,
            pct_start=config["Warmup_Ratio"],
            anneal_strategy='cos',
        )
        
        # Training state
        self.best_val_loss = float('inf')
        self.patience_counter = 0
        self.training_history = {
            'train_loss': [],
            'val_loss': [],
            'lr': [],
            'val_metrics': []
        }
    
    def train_epoch(self, epoch: int) -> float:
        """Train for one epoch using model's built-in DETR loss."""
        self.model.train()
        total_loss = 0.0
        num_batches = 0
        
        pbar = tqdm(self.train_loader, desc=f"Epoch {epoch+1}/{self.config['Epochs']}")
        
        self.optimizer.zero_grad()
        
        for batch_idx, batch in enumerate(pbar):
            pixel_values = batch["pixel_values"].to(self.device)
            labels = [{k: v.to(self.device) for k, v in label.items()} 
                      for label in batch["labels"]]
            
            # Forward pass - model computes DETR loss internally with Hungarian matching
            outputs = self.model(pixel_values=pixel_values, labels=labels)
            
            # Use model's built-in loss (uses Hungarian matching)
            loss = outputs.loss
            
            if loss is None:
                print(f"Warning: Loss is None at batch {batch_idx}")
                continue
            
            # Scale loss for gradient accumulation
            loss = loss / self.config["Gradient_Accumulation"]
            
            # Backward pass
            loss.backward()
            
            # Gradient accumulation
            if (batch_idx + 1) % self.config["Gradient_Accumulation"] == 0:
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                self.optimizer.step()
                self.scheduler.step()
                self.optimizer.zero_grad()
            
            total_loss += loss.item() * self.config["Gradient_Accumulation"]
            num_batches += 1
            
            pbar.set_postfix({
                'loss': f'{loss.item() * self.config["Gradient_Accumulation"]:.4f}',
                'lr': f'{self.scheduler.get_last_lr()[0]:.2e}'
            })
        
        return total_loss / max(num_batches, 1)
    
    @torch.no_grad()
    def validate(self) -> Tuple[float, Dict]:
        """Validate the model using built-in loss and compute metrics."""
        self.model.eval()
        total_loss = 0.0
        num_batches = 0
        
        # Initialize metrics for this validation run
        metrics = TableDetectionMetrics(iou_threshold=self.config["IOU_Threshold"])
        
        for batch in tqdm(self.val_loader, desc="Validating"):
            pixel_values = batch["pixel_values"].to(self.device)
            labels = [{k: v.to(self.device) for k, v in label.items()} 
                      for label in batch["labels"]]
            
            outputs = self.model(pixel_values=pixel_values, labels=labels)
            
            loss = outputs.loss
            if loss is not None:
                total_loss += loss.item()
                num_batches += 1
                
            # Compute metrics
            # Use SAME post-processing as baseline for consistency
            # We need original image sizes for post-processing
            # Assuming batch["labels"] contains original image sizes or we can infer from pixel_values
            # For simplicity, we'll use the padded size (800x800) for validation metrics during training
            # This is an approximation, full evaluation happens after training
            target_sizes = torch.tensor([[self.config["Image_Size"], self.config["Image_Size"]]] * len(pixel_values)).to(self.device)
            
            results = processor.post_process_object_detection(
                outputs, 
                target_sizes=target_sizes,
                threshold=self.config["Conf_Threshold"]
            )
            
            for i, result in enumerate(results):
                pred_boxes = []
                pred_confs = []
                
                # Filter for table class (label 0 in TATR detection model)
                for score, label, box in zip(result["scores"], result["labels"], result["boxes"]):
                    if label.item() == 0:  # Table class
                        pred_boxes.append(box.cpu().numpy())
                        pred_confs.append(float(score.cpu().numpy()))
                
                # Get ground truth boxes
                gt_boxes = []
                if "boxes" in labels[i]:
                    # Convert from [cx, cy, w, h] normalized to [x1, y1, x2, y2] absolute
                    for box in labels[i]["boxes"]:
                        cx, cy, w, h = box.cpu().numpy()
                        x1 = (cx - w/2) * self.config["Image_Size"]
                        y1 = (cy - h/2) * self.config["Image_Size"]
                        x2 = (cx + w/2) * self.config["Image_Size"]
                        y2 = (cy + h/2) * self.config["Image_Size"]
                        gt_boxes.append(np.array([x1, y1, x2, y2]))
                
                metrics.add_batch([pred_boxes], [gt_boxes], [pred_confs])
        
        val_metrics = metrics.get_all_metrics()
        return total_loss / max(num_batches, 1), val_metrics
    
    def save_checkpoint(self, path: str, is_best: bool = False):
        """Save model checkpoint."""
        checkpoint = {
            'model_state_dict': self.model.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'scheduler_state_dict': self.scheduler.state_dict(),
            'best_val_loss': self.best_val_loss,
            'training_history': self.training_history,
        }
        torch.save(checkpoint, path)
        
        if is_best:
            best_path = path.replace('.pt', '_best.pt')
            torch.save(checkpoint, best_path)
    
    def train(self) -> Dict:
        """Full training loop."""
        print("\n" + "="*60)
        print(" TRAINING TATR WITH LoRA")
        print("="*60)
        
        for epoch in range(self.config["Epochs"]):
            # Train
            train_loss = self.train_epoch(epoch)
            self.training_history['train_loss'].append(train_loss)
            self.training_history['lr'].append(self.scheduler.get_last_lr()[0])
            
            # Validate
            val_loss, val_metrics = self.validate()
            self.training_history['val_loss'].append(val_loss)
            self.training_history['val_metrics'].append(val_metrics)
            
            print(f"\nEpoch {epoch+1}/{self.config['Epochs']}: "
                  f"Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")
            print(f"Metrics: F1: {val_metrics['F1-Score']:.4f}, mAP@50: {val_metrics['mAP@50']:.4f}, IoU: {val_metrics['Mean IoU']:.4f}")
            
            # Check for improvement
            if val_loss < self.best_val_loss:
                self.best_val_loss = val_loss
                self.patience_counter = 0
                
                # Save best model
                checkpoint_path = os.path.join(
                    self.config["Output_Dir"], 
                    "checkpoints", 
                    "tatr_lora_best.pt"
                )
                self.save_checkpoint(checkpoint_path, is_best=True)
                print(f"✓ New best model saved (val_loss: {val_loss:.4f})")
            else:
                self.patience_counter += 1
                print(f"  No improvement. Patience: {self.patience_counter}/{self.config['Patience']}")
            
            # Early stopping
            if self.patience_counter >= self.config["Patience"]:
                print(f"\n⚠ Early stopping triggered at epoch {epoch+1}")
                break
            
            # Save periodic checkpoint
            if (epoch + 1) % 5 == 0:
                checkpoint_path = os.path.join(
                    self.config["Output_Dir"], 
                    "checkpoints", 
                    f"tatr_lora_epoch_{epoch+1}.pt"
                )
                self.save_checkpoint(checkpoint_path)
        
        print("\n" + "="*60)
        print(" TRAINING COMPLETE")
        print(f" Best Val Loss: {self.best_val_loss:.4f}")
        print("="*60)
        
        return self.training_history


# Create trainer and run training
trainer = TATRTrainer(
    model=enhanced_model,
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    config=CONFIG
)

# Run training
training_history = trainer.train()

<a id='evaluation'></a>
## 7. Evaluation Metrics Implementation

Evaluate the enhanced TATR model on the test set using:
- **IoU (Intersection over Union)**
- **mAP@50 and mAP@50-95**
- **Precision, Recall, F1-Score**

In [ ]:
# ============================================================
# EVALUATE ENHANCED MODEL
# ============================================================
def evaluate_tatr_model(model: nn.Module, 
                        processor: DetrImageProcessor,
                        test_images: Dict, 
                        test_annotations: Dict,
                        image_dir: str, 
                        config: Dict, 
                        model_name: str = "Model") -> Tuple[Dict, List[float]]:
    """
    Comprehensive evaluation of a TATR model on test set.
    Uses same post-processing as baseline for consistency.
    
    Returns:
        Tuple of (metrics dictionary, list of IoU scores)
    """
    print(f"\n{'='*60}")
    print(f" EVALUATING: {model_name}")
    print(f"{'='*60}")
    
    model.eval()
    metrics = TableDetectionMetrics(iou_threshold=config["IOU_Threshold"])
    
    # Per-image IoU tracking for distribution
    per_image_ious = []
    
    with torch.no_grad():
        for img_id, img_info in tqdm(test_images.items(), desc="Evaluating"):
            img_path = os.path.join(image_dir, img_info['file_name'])
            
            if not os.path.exists(img_path):
                continue
            
            # Load and preprocess image
            image = Image.open(img_path).convert("RGB")
            inputs = processor(images=image, return_tensors="pt")
            inputs = {k: v.to(config["Device"]) for k, v in inputs.items()}
            
            # Ground truth
            gt_anns = test_annotations.get(img_id, [])
            gt_boxes = [coco_bbox_to_xyxy(ann['bbox'], img_info['width'], img_info['height']) 
                        for ann in gt_anns]
            
            # Get model output
            outputs = model(**inputs)
            
            # Use SAME post-processing as baseline for consistency
            target_sizes = torch.tensor([[img_info['height'], img_info['width']]]).to(config["Device"])
            results = processor.post_process_object_detection(
                outputs, 
                target_sizes=target_sizes,
                threshold=config["Conf_Threshold"]
            )[0]
            
            pred_boxes = []
            pred_confs = []
            
            # Filter for table class (label 0 in TATR detection model)
            for score, label, box in zip(results["scores"], results["labels"], results["boxes"]):
                if label.item() == 0:  # Table class
                    pred_boxes.append(box.cpu().numpy())
                    pred_confs.append(float(score.cpu().numpy()))
            
            # Calculate per-image IoU
            if pred_boxes and gt_boxes:
                for pred in pred_boxes:
                    max_iou = max(metrics.calculate_iou(pred, gt) for gt in gt_boxes)
                    per_image_ious.append(max_iou)
            
            metrics.add_batch(pred_boxes, gt_boxes, pred_confs)
    
    # Get and print results
    results_summary = metrics.print_summary(f"{model_name} Evaluation Results")
    
    # Add IoU distribution stats
    if per_image_ious:
        results_summary["IoU_Std"] = float(np.std(per_image_ious))
        results_summary["IoU_Min"] = float(np.min(per_image_ious))
        results_summary["IoU_Max"] = float(np.max(per_image_ious))
    
    return results_summary, per_image_ious


# Load best model checkpoint
best_model_path = os.path.join(CONFIG["Output_Dir"], "checkpoints", "tatr_lora_best.pt")

if os.path.exists(best_model_path):
    checkpoint = torch.load(best_model_path, map_location=CONFIG["Device"])
    enhanced_model.load_state_dict(checkpoint['model_state_dict'])
    print(f"✓ Loaded best model from: {best_model_path}")
else:
    print(f"⚠ Best model not found at {best_model_path}, using current model")

# Evaluate enhanced model
enhanced_metrics, enhanced_ious = evaluate_tatr_model(
    model=enhanced_model,
    processor=processor,
    test_images=test_images,
    test_annotations=test_annotations,
    image_dir=CONFIG["Image_Dir"],
    config=CONFIG,
    model_name="Enhanced TATR (LoRA)"
)

<a id='comparison'></a>
## 8. Comparison: Before vs After Enhancement

Compare baseline TATR with the enhanced TATR (with LoRA) across all evaluation metrics.

In [ ]:
# ============================================================
# RE-RUN BASELINE FOR FAIR COMPARISON
# ============================================================
# Load fresh baseline model
print("Loading baseline model for comparison...")

baseline_model_fresh = TableTransformerForObjectDetection.from_pretrained(
    CONFIG["Base_Model"]
).to(CONFIG["Device"])
baseline_model_fresh.eval()

baseline_processor_fresh = DetrImageProcessor.from_pretrained(CONFIG["Base_Model"])

# Evaluate baseline
baseline_metrics_final, baseline_ious = evaluate_tatr_model(
    model=baseline_model_fresh,
    processor=baseline_processor_fresh,
    test_images=test_images,
    test_annotations=test_annotations,
    image_dir=CONFIG["Image_Dir"],
    config=CONFIG,
    model_name="Baseline TATR"
)

In [ ]:
# ============================================================
# COMPREHENSIVE COMPARISON TABLE
# ============================================================
def create_comparison_table(baseline: Dict, enhanced: Dict) -> pd.DataFrame:
    """Create formatted comparison DataFrame."""
    metrics_to_compare = [
        ("Precision", "Precision"),
        ("Recall", "Recall"),
        ("F1-Score", "F1-Score"),
        ("Mean IoU", "Mean IoU"),
        ("mAP@50", "mAP@50"),
        ("mAP@50-95", "mAP@50-95"),
    ]
    
    data = []
    for display_name, key in metrics_to_compare:
        baseline_val = baseline.get(key, 0)
        enhanced_val = enhanced.get(key, 0)
        improvement = enhanced_val - baseline_val
        pct_change = (improvement / baseline_val * 100) if baseline_val > 0 else 0
        
        data.append({
            "Metric": display_name,
            "Baseline TATR": f"{baseline_val:.4f}",
            "Enhanced TATR (LoRA)": f"{enhanced_val:.4f}",
            "Improvement": f"{improvement:+.4f}",
            "Change (%)": f"{pct_change:+.2f}%"
        })
    
    return pd.DataFrame(data)

comparison_df = create_comparison_table(baseline_metrics_final, enhanced_metrics)

print("\n" + "="*80)
print(" COMPARISON: BASELINE vs ENHANCED TATR (LoRA)")
print("="*80)
print(comparison_df.to_string(index=False))
print("="*80)

In [ ]:
# ============================================================
# VISUALIZATION: METRICS COMPARISON
# ============================================================
def plot_metrics_comparison(baseline: Dict, enhanced: Dict, save_path: str = None):
    """
    Create visualization comparing baseline and enhanced model metrics.
    """
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('TATR Table Detection: Baseline vs Enhanced (LoRA)', fontsize=14, fontweight='bold')
    
    # Color palette
    colors = ['#3498db', '#2ecc71']
    
    # 1. Main Metrics Bar Chart
    ax1 = axes[0, 0]
    metrics = ['Precision', 'Recall', 'F1-Score', 'Mean IoU']
    baseline_vals = [baseline.get(m, 0) for m in metrics]
    enhanced_vals = [enhanced.get(m, 0) for m in metrics]
    
    x = np.arange(len(metrics))
    width = 0.35
    
    bars1 = ax1.bar(x - width/2, baseline_vals, width, label='Baseline', color=colors[0], alpha=0.8)
    bars2 = ax1.bar(x + width/2, enhanced_vals, width, label='Enhanced (LoRA)', color=colors[1], alpha=0.8)
    
    ax1.set_ylabel('Score')
    ax1.set_title('Detection Metrics Comparison')
    ax1.set_xticks(x)
    ax1.set_xticklabels(metrics)
    ax1.legend()
    ax1.set_ylim(0, 1.1)
    ax1.grid(axis='y', alpha=0.3)
    
    # Add value labels
    for bar in bars1:
        height = bar.get_height()
        ax1.annotate(f'{height:.3f}', xy=(bar.get_x() + bar.get_width()/2, height),
                    xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=8)
    for bar in bars2:
        height = bar.get_height()
        ax1.annotate(f'{height:.3f}', xy=(bar.get_x() + bar.get_width()/2, height),
                    xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=8)
    
    # 2. mAP Comparison
    ax2 = axes[0, 1]
    map_metrics = ['mAP@50', 'mAP@50-95']
    baseline_map = [baseline.get(m, 0) for m in map_metrics]
    enhanced_map = [enhanced.get(m, 0) for m in map_metrics]
    
    x = np.arange(len(map_metrics))
    bars1 = ax2.bar(x - width/2, baseline_map, width, label='Baseline', color=colors[0], alpha=0.8)
    bars2 = ax2.bar(x + width/2, enhanced_map, width, label='Enhanced (LoRA)', color=colors[1], alpha=0.8)
    
    ax2.set_ylabel('mAP')
    ax2.set_title('Mean Average Precision Comparison')
    ax2.set_xticks(x)
    ax2.set_xticklabels(map_metrics)
    ax2.legend()
    ax2.set_ylim(0, 1.1)
    ax2.grid(axis='y', alpha=0.3)
    
    for bar in bars1:
        height = bar.get_height()
        ax2.annotate(f'{height:.3f}', xy=(bar.get_x() + bar.get_width()/2, height),
                    xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=8)
    for bar in bars2:
        height = bar.get_height()
        ax2.annotate(f'{height:.3f}', xy=(bar.get_x() + bar.get_width()/2, height),
                    xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=8)
    
    # 3. Improvement Chart
    ax3 = axes[1, 0]
    all_metrics = ['Precision', 'Recall', 'F1-Score', 'Mean IoU', 'mAP@50', 'mAP@50-95']
    improvements = [
        (enhanced.get(m, 0) - baseline.get(m, 0)) * 100 
        for m in all_metrics
    ]
    
    colors_imp = ['#2ecc71' if v >= 0 else '#e74c3c' for v in improvements]
    bars = ax3.barh(all_metrics, improvements, color=colors_imp, alpha=0.8)
    ax3.axvline(x=0, color='black', linewidth=0.5)
    ax3.set_xlabel('Improvement (%)')
    ax3.set_title('Percentage Improvement (Enhanced vs Baseline)')
    ax3.grid(axis='x', alpha=0.3)
    
    for bar, val in zip(bars, improvements):
        width_val = bar.get_width()
        ax3.annotate(f'{val:+.2f}%', 
                    xy=(width_val, bar.get_y() + bar.get_height()/2),
                    xytext=(5 if width_val >= 0 else -5, 0), 
                    textcoords="offset points",
                    ha='left' if width_val >= 0 else 'right', 
                    va='center', fontsize=9)
    
    # 4. TP/FP/FN Comparison
    ax4 = axes[1, 1]
    categories = ['True Positives', 'False Positives', 'False Negatives']
    baseline_counts = [baseline.get('TP', 0), baseline.get('FP', 0), baseline.get('FN', 0)]
    enhanced_counts = [enhanced.get('TP', 0), enhanced.get('FP', 0), enhanced.get('FN', 0)]
    
    x = np.arange(len(categories))
    bars1 = ax4.bar(x - width/2, baseline_counts, width, label='Baseline', color=colors[0], alpha=0.8)
    bars2 = ax4.bar(x + width/2, enhanced_counts, width, label='Enhanced (LoRA)', color=colors[1], alpha=0.8)
    
    ax4.set_ylabel('Count')
    ax4.set_title('Detection Statistics')
    ax4.set_xticks(x)
    ax4.set_xticklabels(categories)
    ax4.legend()
    ax4.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"✓ Comparison plot saved: {save_path}")
    
    plt.show()
    return fig

# Create comparison visualization
plot_path = os.path.join(CONFIG["Output_Dir"], "comparison_plot.png")
fig = plot_metrics_comparison(baseline_metrics_final, enhanced_metrics, plot_path)

In [ ]:
# ============================================================
# IoU DISTRIBUTION COMPARISON
# ============================================================
def plot_iou_distribution(baseline_ious: List[float], enhanced_ious: List[float], 
                          save_path: str = None):
    """Plot IoU distribution comparison between baseline and enhanced models."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # 1. Histogram
    ax1 = axes[0]
    if baseline_ious:
        ax1.hist(baseline_ious, bins=30, alpha=0.6, label='Baseline', color='#3498db', density=True)
        ax1.axvline(x=np.mean(baseline_ious), color='#3498db', linestyle='--', linewidth=2,
                    label=f'Baseline Mean: {np.mean(baseline_ious):.3f}')
    if enhanced_ious:
        ax1.hist(enhanced_ious, bins=30, alpha=0.6, label='Enhanced (LoRA)', color='#2ecc71', density=True)
        ax1.axvline(x=np.mean(enhanced_ious), color='#2ecc71', linestyle='--', linewidth=2,
                    label=f'Enhanced Mean: {np.mean(enhanced_ious):.3f}')
    ax1.set_xlabel('IoU Score')
    ax1.set_ylabel('Density')
    ax1.set_title('IoU Distribution Comparison')
    ax1.legend()
    ax1.grid(alpha=0.3)
    
    # 2. Box plot
    ax2 = axes[1]
    data_to_plot = []
    labels_to_plot = []
    if baseline_ious:
        data_to_plot.append(baseline_ious)
        labels_to_plot.append('Baseline')
    if enhanced_ious:
        data_to_plot.append(enhanced_ious)
        labels_to_plot.append('Enhanced (LoRA)')
    
    if data_to_plot:
        bp = ax2.boxplot(data_to_plot, labels=labels_to_plot, patch_artist=True)
        colors_box = ['#3498db', '#2ecc71'][:len(data_to_plot)]
        for box, color in zip(bp['boxes'], colors_box):
            box.set_facecolor(color)
            box.set_alpha(0.6)
    
    ax2.set_ylabel('IoU Score')
    ax2.set_title('IoU Distribution Box Plot')
    ax2.grid(axis='y', alpha=0.3)
    
    # Add statistics
    stats_parts = []
    if baseline_ious:
        stats_parts.append(f"Baseline: μ={np.mean(baseline_ious):.3f}, σ={np.std(baseline_ious):.3f}")
    if enhanced_ious:
        stats_parts.append(f"Enhanced: μ={np.mean(enhanced_ious):.3f}, σ={np.std(enhanced_ious):.3f}")
    
    if stats_parts:
        stats_text = "\n".join(stats_parts)
        ax2.text(0.5, -0.15, stats_text, transform=ax2.transAxes, ha='center', 
                 fontsize=10, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"✓ IoU distribution plot saved: {save_path}")
    
    plt.show()
    return fig

# Plot IoU distributions (if we have data)
if baseline_ious or enhanced_ious:
    iou_plot_path = os.path.join(CONFIG["Output_Dir"], "iou_distribution.png")
    plot_iou_distribution(baseline_ious, enhanced_ious, iou_plot_path)

In [ ]:
# ============================================================
# FINAL SUMMARY REPORT
# ============================================================
def generate_summary_report(baseline: Dict, enhanced: Dict, config: Dict):
    """Generate final summary report."""
    print("\n" + "="*80)
    print(" FINAL SUMMARY REPORT: TATR TABLE DETECTION WITH LoRA")
    print("="*80)
    
    print("\n📊 EVALUATION CRITERIA RESULTS")
    print("-"*60)
    
    # IoU
    print("\n1️⃣ INTERSECTION OVER UNION (IoU)")
    print(f"   Baseline Mean IoU:  {baseline.get('Mean IoU', 0):.4f}")
    print(f"   Enhanced Mean IoU:  {enhanced.get('Mean IoU', 0):.4f}")
    iou_imp = enhanced.get('Mean IoU', 0) - baseline.get('Mean IoU', 0)
    baseline_iou = baseline.get('Mean IoU', 1)
    print(f"   Improvement:        {iou_imp:+.4f} ({iou_imp/baseline_iou*100 if baseline_iou > 0 else 0:+.2f}%)")
    
    # mAP
    print("\n2️⃣ MEAN AVERAGE PRECISION (mAP)")
    print(f"   Baseline mAP@50:    {baseline.get('mAP@50', 0):.4f}")
    print(f"   Enhanced mAP@50:    {enhanced.get('mAP@50', 0):.4f}")
    map50_imp = enhanced.get('mAP@50', 0) - baseline.get('mAP@50', 0)
    baseline_map50 = baseline.get('mAP@50', 1)
    print(f"   Improvement:        {map50_imp:+.4f} ({map50_imp/baseline_map50*100 if baseline_map50 > 0 else 0:+.2f}%)")
    print()
    print(f"   Baseline mAP@50-95: {baseline.get('mAP@50-95', 0):.4f}")
    print(f"   Enhanced mAP@50-95: {enhanced.get('mAP@50-95', 0):.4f}")
    map5095_imp = enhanced.get('mAP@50-95', 0) - baseline.get('mAP@50-95', 0)
    baseline_map5095 = baseline.get('mAP@50-95', 1)
    print(f"   Improvement:        {map5095_imp:+.4f} ({map5095_imp/baseline_map5095*100 if baseline_map5095 > 0 else 0:+.2f}%)")
    
    # F1-Score
    print("\n3️⃣ F1-SCORE")
    print(f"   Baseline F1-Score:  {baseline.get('F1-Score', 0):.4f}")
    print(f"   Enhanced F1-Score:  {enhanced.get('F1-Score', 0):.4f}")
    f1_imp = enhanced.get('F1-Score', 0) - baseline.get('F1-Score', 0)
    baseline_f1 = baseline.get('F1-Score', 1)
    print(f"   Improvement:        {f1_imp:+.4f} ({f1_imp/baseline_f1*100 if baseline_f1 > 0 else 0:+.2f}%)")
    
    # Architecture details
    print("\n🔧 LoRA CONFIGURATION")
    print("-"*60)
    print(f"   • LoRA Rank (r): {config['LoRA_r']}")
    print(f"   • LoRA Alpha: {config['LoRA_alpha']}")
    print(f"   • LoRA Dropout: {config['LoRA_dropout']}")
    print(f"   • Base Model: {config['Base_Model']}")
    
    # Training details
    print("\n⚙️ TRAINING CONFIGURATION")
    print("-"*60)
    print(f"   • Image Size: {config['Image_Size']}x{config['Image_Size']}")
    print(f"   • Batch Size: {config['Batch_Size']} (Effective: {config['Batch_Size'] * config['Gradient_Accumulation']})")
    print(f"   • Epochs: {config['Epochs']}")
    print(f"   • Learning Rate: {config['Learning_Rate']}")
    print(f"   • Early Stopping Patience: {config['Patience']}")
    
    print("\n" + "="*80)
    print(" CONCLUSION")
    print("="*80)
    
    # Determine if enhancement was successful
    improvements = {
        'IoU': iou_imp,
        'mAP@50': map50_imp,
        'mAP@50-95': map5095_imp,
        'F1-Score': f1_imp
    }
    
    positive_improvements = sum(1 for v in improvements.values() if v > 0)
    
    if positive_improvements >= 3:
        print("✅ The LoRA fine-tuned TATR shows SIGNIFICANT IMPROVEMENT")
        print("   over the baseline model across multiple metrics.")
    elif positive_improvements >= 2:
        print("⚠️ The LoRA fine-tuned TATR shows MODERATE IMPROVEMENT")
        print("   with gains in some metrics but not all.")
    elif baseline.get('F1-Score', 0) > 0.95:
        print("ℹ️ Baseline model is already EXCELLENT (F1 > 0.95).")
        print("   Fine-tuning may not provide additional benefit.")
        print("   Consider using baseline model directly.")
    else:
        print("❌ Fine-tuning did not improve over baseline.")
        print("   Consider: lower LR, fewer epochs, or using baseline as-is.")
    
    print("\n" + "="*80 + "\n")

# Generate final report
generate_summary_report(baseline_metrics_final, enhanced_metrics, CONFIG)

In [ ]:
# ============================================================
# SAVE RESULTS
# ============================================================
# Save all results for later analysis
results = {
    'baseline_metrics': baseline_metrics_final,
    'enhanced_metrics': enhanced_metrics,
    'config': CONFIG,
    'timestamp': str(datetime.now()),
    'model_type': 'TATR with LoRA',
    'architecture': {
        'base_model': CONFIG['Base_Model'],
        'lora': {
            'r': CONFIG['LoRA_r'],
            'alpha': CONFIG['LoRA_alpha'],
            'dropout': CONFIG['LoRA_dropout'],
            'target_modules': CONFIG['LoRA_target_modules']
        }
    }
}

# Save to JSON
results_path = 'TATR_LoRA_training_results.json'
with open(results_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f"✅ Results saved to: {results_path}")

# Save the enhanced model if training was successful
if enhanced_model is not None:
    enhanced_model_path = 'TATR_LoRA_enhanced_table_detection'
    enhanced_model.save_pretrained(enhanced_model_path)
    print(f"✅ Enhanced model saved to: {enhanced_model_path}/")
    
    # Also save the processor
    processor_path = os.path.join(enhanced_model_path, 'processor')
    processor.save_pretrained(processor_path)
    print(f"✅ Processor saved to: {processor_path}/")

## Summary

This notebook demonstrated a complete pipeline for **Table Detection using TATR (Table Transformer)** with **LoRA (Low-Rank Adaptation)** fine-tuning.

### Key Components:

1. **Baseline Model**: Microsoft's Table Transformer (`microsoft/table-transformer-detection`)
2. **LoRA Fine-tuning**: Parameter-efficient training targeting attention layers (~2% trainable params)
3. **Augmentation**: Geometric and photometric transforms from augmentation_config.json

### Evaluation Metrics:
- **IoU (Intersection over Union)**: Measures overlap between predicted and ground truth boxes
- **mAP@50**: Mean Average Precision at IoU threshold 0.5
- **mAP@50-95**: Mean Average Precision averaged across IoU thresholds 0.5 to 0.95
- **F1-Score**: Harmonic mean of precision and recall

### LoRA Configuration:
- Rank (r): 8 (conservative for fine-tuning)
- Alpha: 16
- Target modules: Self-attention and encoder-attention layers

### Important Notes:
- **Baseline is often excellent**: TATR pre-trained model already achieves F1 > 0.99 on many datasets
- **Fine-tuning may not help**: If baseline is near-perfect, LoRA fine-tuning may not improve (or could hurt)
- **Use conservative settings**: Low LR (1e-5), few epochs (15), small LoRA rank (8)

### Next Steps:
- If baseline F1 > 0.95, consider using baseline model directly
- For challenging datasets, fine-tuning with augmentation may help
- Integrate with table structure recognition for complete table extraction